# LAB | Hyperparameter Tuning

**Load the data**

Finally step in order to maximize the performance on your Spaceship Titanic model.

The data can be found here:

https://raw.githubusercontent.com/data-bootcamp-v4/data/main/spaceship_titanic.csv

Metadata

https://github.com/data-bootcamp-v4/data/blob/main/spaceship_titanic.md

So far we've been training and evaluating models with default values for hyperparameters.

Today we will perform the same feature engineering as before, and then compare the best working models you got so far, but now fine tuning it's hyperparameters.

In [ ]:
#Libraries
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectKBest, f_classif

In [ ]:
spaceship = pd.read_csv("https://raw.githubusercontent.com/data-bootcamp-v4/data/main/spaceship_titanic.csv")
spaceship.head()

Now perform the same as before:
- Feature Scaling
- Feature Selection


In [ ]:
# Limpieza básica
spaceship = spaceship.dropna()

# Separar Cabin en 3 columnas (Deck / Num / Side)
spaceship[["Deck", "Cabin_num", "Side"]] = spaceship["Cabin"].str.split("/", expand=True)

# Convertir a numérico donde proceda
spaceship["Cabin_num"] = pd.to_numeric(spaceship["Cabin_num"], errors="coerce")

# Codificación dummies
spaceship = pd.get_dummies(spaceship, drop_first=True)

# Target y features
y = spaceship["Transported"]
X = spaceship.drop("Transported", axis=1)

In [ ]:
# Feature Scaling
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

In [ ]:
# Feature Selection
selector = SelectKBest(score_func=f_classif, k=10)
X_selected = selector.fit_transform(X_scaled, y)

selected_features = X.columns[selector.get_support()]
selected_features


- Now let's use the best model we got so far in order to see how it can improve when we fine tune it's hyperparameters.

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

X_train, X_test, y_train, y_test = train_test_split(X_selected, y, test_size=0.2, random_state=42)

base_model = RandomForestClassifier(random_state=42)
base_model.fit(X_train, y_train)

y_pred = base_model.predict(X_test)
acc_base = accuracy_score(y_test, y_pred)
acc_base


In [ ]:
from sklearn.model_selection import GridSearchCV

param_grid = {
    "n_estimators": [200, 400, 600],
    "max_depth": [None, 10, 20, 30],
    "min_samples_split": [2, 5, 10],
    "min_samples_leaf": [1, 2, 4],
    "bootstrap": [True, False]
}

rf = RandomForestClassifier(random_state=42)

grid = GridSearchCV(
    estimator=rf,
    param_grid=param_grid,
    cv=3,
    scoring="accuracy",
    n_jobs=-1
)

grid.fit(X_train, y_train)

grid.best_params_, grid.best_score_


- Evaluate your model

In [ ]:
best_rf = grid.best_estimator_

y_pred_best = best_rf.predict(X_test)
acc_best = accuracy_score(y_test, y_pred_best)

acc_best


**Grid/Random Search**

For this lab we will use Grid Search.

- Define hyperparameters to fine tune.

In [ ]:
rf = RandomForestClassifier(random_state=42)

param_grid = {
    "n_estimators": [100, 300, 500],        # nº de árboles
    "max_depth": [None, 10, 20, 30],        # profundidad máxima
    "min_samples_split": [2, 5, 10],        # min muestras para dividir un nodo
    "min_samples_leaf": [1, 2, 4],          # min muestras en una hoja
    "max_features": ["sqrt", "log2"],       # nº de features por split
    "bootstrap": [True, False]              # muestreo con/sin reemplazo
}

- Run Grid Search

In [ ]:
grid = GridSearchCV(
    estimator=rf,
    param_grid=param_grid,
    cv=3,
    scoring="accuracy",
    n_jobs=-1
)

grid.fit(X_train, y_train)

grid.best_params
grid.best_score


- Evaluate your model

In [ ]:
from sklearn.metrics import accuracy_score, classification_report

# Mejor modelo encontrado
best_model = grid.best_estimator

# Predicciones sobre TEST
y_pred = best_model.predict(X_test)

# Accuracy
accuracy = accuracy_score(y_test, y_pred)
print("Test Accuracy:", accuracy)

# Informe de clasificación
print(classification_report(y_test, y_pred))
